In [1]:
!pip install python-binance pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 47.5 MB/s eta 0:00:00


In [2]:
import os
import sys
import logging
from google.colab import userdata
from pydantic import BaseModel, Field, field_validator, model_validator
from binance.client import Client
from binance.exceptions import BinanceAPIException, BinanceRequestException

# ==========================================
# 1. SETUP LOGGING & SECURE KEYS
# ==========================================
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/bot.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True
)
logger = logging.getLogger("TradingBot")

# Fetch keys from Colab Secrets (🔑 sidebar)
try:
    API_KEY = userdata.get('BINANCE_API_KEY')
    API_SECRET = userdata.get('BINANCE_API_SECRET')
    logger.info("Successfully loaded API credentials from Colab Secrets.")
except Exception as e:
    print("❌ ERROR: Could not retrieve keys from Colab Secrets.")
    print("Please check the 🔑 Secrets tab and ensure 'Notebook access' is turned ON.")
    logger.error("Failed to retrieve API credentials from Colab Secrets.")

# ==========================================
# 2. VALIDATION LAYER (Pydantic)
# ==========================================
class OrderInput(BaseModel):
    symbol: str
    side: str
    order_type: str
    quantity: float = Field(..., gt=0, description="Quantity must be greater than zero")
    price: float | None = None

    @field_validator('symbol')
    @classmethod
    def validate_symbol(cls, v: str) -> str:
        v = v.upper().strip()
        if not v.endswith("USDT"):
            raise ValueError("Only USDT-M trading pairs are supported (e.g., BTCUSDT, ETHUSDT).")
        return v

    @field_validator('side')
    @classmethod
    def validate_side(cls, v: str) -> str:
        v = v.upper().strip()
        if v not in ['BUY', 'SELL']:
            raise ValueError("Side must be either 'BUY' or 'SELL'.")
        return v

    @field_validator('order_type')
    @classmethod
    def validate_order_type(cls, v: str) -> str:
        v = v.upper().strip()
        if v not in ['MARKET', 'LIMIT']:
            raise ValueError("Order type must be 'MARKET' or 'LIMIT'.")
        return v

    @model_validator(mode='after')
    def validate_price_for_limit(self):
        if self.order_type == 'LIMIT' and self.price is None:
            raise ValueError("Price is strictly required for LIMIT orders.")
        return self

# ==========================================
# 3. BINANCE FUTURES CLIENT LAYER
# ==========================================
class BinanceFuturesTestnetBot:
    def __init__(self, api_key: str, api_secret: str):
        if not api_key or not api_secret:
            raise ValueError("API Key and API Secret cannot be empty.")

        # testnet=True targets https://testnet.binancefuture.com
        self.client = Client(api_key, api_secret, testnet=True)
        logger.info("Initialized Binance Futures Testnet client.")

    def place_order(self, order_data: OrderInput):
        payload = {
            "symbol": order_data.symbol,
            "side": order_data.side,
            "type": order_data.order_type,
            "quantity": order_data.quantity
        }

        if order_data.order_type == "LIMIT":
            payload["price"] = str(order_data.price)
            payload["timeInForce"] = "GTC"  # Good 'Til Cancelled

        logger.info(f"Sending API payload to Testnet: {payload}")

        try:
            # Execute on Binance USDT-M Futures endpoint
            response = self.client.futures_create_order(**payload)
            logger.info(f"Execution Success. Raw Response: {response}")
            return {"success": True, "data": response}

        except BinanceAPIException as e:
            logger.error(f"Binance API Exception [Code {e.status_code}]: {e.message}")
            return {"success": False, "error": f"API Error ({e.status_code}): {e.message}"}
        except BinanceRequestException as e:
            logger.error(f"Network request error: {str(e)}")
            return {"success": False, "error": f"Network Error: {str(e)}"}
        except Exception as e:
            logger.error(f"Unexpected Exception: {str(e)}")
            return {"success": False, "error": f"Unexpected System Error: {str(e)}"}

# ==========================================
# 4. MAIN EXECUTION CONTROLLER
# ==========================================
def execute_trade(symbol: str, side: str, order_type: str, quantity: float, price: float = None):
    """
    Controller function that orchestrates validation, client connection, and logging output.
    """
    print("\n==================================================")
    print("          ORDER REQUEST SUMMARY                   ")
    print("==================================================")
    print(f"Target Symbol : {symbol}")
    print(f"Order Side    : {side}")
    print(f"Order Type    : {order_type}")
    print(f"Quantity      : {quantity}")
    print(f"Price         : {price if price else 'N/A (Market Order)'}")
    print("--------------------------------------------------")

    # 1. Validate Input
    try:
        validated_input = OrderInput(
            symbol=symbol,
            side=side,
            order_type=order_type,
            quantity=quantity,
            price=price
        )
    except Exception as e:
        print(f"\n❌ VALIDATION ERROR:\n{e}\n")
        logger.error(f"Input validation failure for request: {e}")
        return

    # 2. Initialize Client
    try:
        bot = BinanceFuturesTestnetBot(API_KEY, API_SECRET)
    except Exception as e:
        print(f"\n❌ INITIALIZATION ERROR: {e}\n")
        return

    # 3. Submit Order
    print("Status: Transmitting order payload to Binance Testnet...")
    result = bot.place_order(validated_input)

    # 4. Print Clean Response
    if result["success"]:
        res = result["data"]
        print("\n SUCCESS: ORDER EXECUTED ON TESTNET")
        print("--------------------------------------------------")
        print(f"Order ID     : {res.get('orderId')}")
        print(f"Status       : {res.get('status')}")
        print(f"Executed Qty : {res.get('executedQty')}")
        print(f"Avg Price    : {res.get('avgPrice', '0.00')}")
        print("--------------------------------------------------")
        print("Detailed logs saved to 'logs/bot.log'\n")
    else:
        print("\n❌ EXECUTION FAILED")
        print(f"Reason: {result['error']}\n")

In [3]:
# 1. Test MARKET Order
execute_trade(symbol="BTCUSDT", side="BUY", order_type="MARKET", quantity=0.01)

# 2. Test LIMIT Order
execute_trade(symbol="BTCUSDT", side="SELL", order_type="LIMIT", quantity=0.01, price=65000)

# 3. Test Invalid Input Handling (Missing price for Limit order)
execute_trade(symbol="BTCUSDT", side="BUY", order_type="LIMIT", quantity=0.01)


          ORDER REQUEST SUMMARY                   
Target Symbol : BTCUSDT
Order Side    : BUY
Order Type    : MARKET
Quantity      : 0.01
Price         : N/A (Market Order)
--------------------------------------------------

❌ INITIALIZATION ERROR: APIError(code=0): Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error.


          ORDER REQUEST SUMMARY                   
Target Symbol : BTCUSDT
Order Side    : SELL
Order Type    : LIMIT
Quantity      : 0.01
Price         : 65000
--------------------------------------------------

❌ INITIALIZATION ERROR: APIError(code=0): Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error.


          ORDER REQUEST SUMMARY                   
Target Symbol : BTCUSDT
Ord